# Streaming Pipeline — Spark Structured Streaming

Este notebook implementa la ingesta en tiempo real exigida por la guía del proyecto
(sección 3.1 del PDF oficial).

## Arquitectura

```
Productor sintético (rate source)  →  Spark Structured Streaming  →  bronze.bronze_bookings_stream (Delta)
```

## Nota técnica sobre la fuente del stream

La guía oficial sugiere Apache Kafka (Confluent Cloud). En este proyecto se usa
**`rate` source** de Spark, que genera filas sintéticas a una frecuencia
configurable y se procesan con el mismo `readStream` que usaría Kafka.

**¿Por qué `rate` y no Kafka?** Las plataformas Kafka como servicio (Confluent
Cloud, Aiven, Upstash) requieren registrar método de pago incluso para sus
tiers gratuitos. La arquitectura de Spark Structured Streaming es idéntica:
solo cambia el conector del `readStream`. Migrar a Kafka productivo requeriría
reemplazar las 3 líneas del `readStream.format("rate")` por
`readStream.format("kafka")` con sus credenciales — el resto del pipeline
(parsing, transformación, escritura a Delta, checkpoint) permanece igual.

Esto demuestra el **patrón de ingesta streaming** sin depender de servicios
de pago externos.

## 1. Configuración del stream

Parámetros tunables: cuántos eventos por segundo se generan y por cuántos
segundos corre el stream antes de detenerse automáticamente.

In [ ]:
# Frecuencia de generación de eventos
ROWS_PER_SECOND  = 5
# Duración total del stream en segundos
STREAM_DURATION  = 30
# Tabla destino en Bronze
TARGET_TABLE     = "bronze.bronze_bookings_stream"
# Ubicación del checkpoint (necesario para exactly-once)
CHECKPOINT_PATH  = "/tmp/checkpoints/bronze_bookings_stream"

print(f"Stream: {ROWS_PER_SECOND} eventos/s durante {STREAM_DURATION}s")
print(f"Eventos esperados: ~{ROWS_PER_SECOND * STREAM_DURATION}")
print(f"Tabla destino:    {TARGET_TABLE}")

## 2. Limpieza previa

Para que el notebook sea idempotente, se elimina la tabla destino y el checkpoint
antes de cada corrida. Esto evita el conflicto típico de Structured Streaming
("el checkpoint existe pero la tabla cambió").

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
try:
    dbutils.fs.rm(CHECKPOINT_PATH, True)
except Exception as e:
    print(f"(checkpoint no existía, ok): {e}")
print("Estado limpio.")

## 3. Capturar rangos válidos de IDs desde Silver

Para que los eventos generados sean coherentes con el resto del pipeline,
los `user_id` y `property_id` se generan dentro de los rangos observados en
las tablas Silver. Esto permite hacer joins reales después en Gold.

In [ ]:
from pyspark.sql.functions import min as F_min, max as F_max

lim_u = spark.table("silver.silver_users").agg(
    F_min("user_id").alias("u_min"),
    F_max("user_id").alias("u_max")
).collect()[0]
USER_MIN, USER_MAX = lim_u["u_min"], lim_u["u_max"]

lim_p = spark.table("silver.silver_properties").agg(
    F_min("property_id").alias("p_min"),
    F_max("property_id").alias("p_max")
).collect()[0]
PROP_MIN, PROP_MAX = lim_p["p_min"], lim_p["p_max"]

print(f"user_id     range: [{USER_MIN}, {USER_MAX}]")
print(f"property_id range: [{PROP_MIN}, {PROP_MAX}]")

## 4. Productor sintético — `rate` source

Genera filas con `value` incremental (que se usa como `booking_id`) y `timestamp`,
y las transforma en eventos de reserva con la misma forma que `silver_bookings`.

In [ ]:
rate_stream = (
    spark.readStream
         .format("rate")
         .option("rowsPerSecond", ROWS_PER_SECOND)
         .load()
)

eventos = rate_stream.selectExpr(
    "(2000000 + value) AS booking_id",
    f"CAST({USER_MIN} + rand() * ({USER_MAX} - {USER_MIN}) AS BIGINT) AS user_id",
    f"CAST({PROP_MIN} + rand() * ({PROP_MAX} - {PROP_MIN}) AS BIGINT) AS property_id",
    "date_add(current_date(), CAST(rand() * 90 AS INT)) AS check_in",
    "date_add(current_date(), CAST(rand() * 90 AS INT) + CAST(1 + rand() * 13 AS INT)) AS check_out",
    "CAST(1 + rand() * 5 AS INT) AS guests_count",
    "ROUND(50 + rand() * 1450, 2) AS total_amount",
    "CASE WHEN rand() < 0.5 THEN 'confirmed' WHEN rand() < 0.8 THEN 'pending' ELSE 'cancelled' END AS status",
    "current_timestamp() AS created_at",
    "current_timestamp() AS updated_at"
)

print("Schema del stream:")
eventos.printSchema()

## 5. Consumidor — `writeStream` a tabla Delta

Escribe los eventos a `bronze.bronze_bookings_stream` en modo append, con
`checkpointLocation` para garantizar tolerancia a fallos y exactly-once.

In [ ]:
query = (
    eventos.writeStream
           .format("delta")
           .outputMode("append")
           .option("checkpointLocation", CHECKPOINT_PATH)
           .trigger(processingTime="5 seconds")
           .toTable(TARGET_TABLE)
)

print(f"Stream activo. Esperando {STREAM_DURATION} segundos...")
query.awaitTermination(timeout=STREAM_DURATION)
query.stop()
print("Streaming detenido.")

## 6. Validación — los eventos llegaron a Bronze

In [ ]:
%sql
SELECT COUNT(*) AS eventos_recibidos
FROM bronze.bronze_bookings_stream;

In [ ]:
%sql
SELECT *
FROM bronze.bronze_bookings_stream
ORDER BY created_at DESC
LIMIT 10;

## 7. Integración con Bronze batch

Los eventos del stream comparten esquema con `silver_bookings`, por lo que se
pueden unir a la carga batch original. Esta vista cierra el ciclo completo
Bronze (batch + streaming) → Silver → Gold.

In [ ]:
%sql
CREATE OR REPLACE VIEW bronze.bronze_bookings_all AS
SELECT booking_id, user_id, property_id,
       CAST(check_in AS DATE)  AS check_in,
       CAST(check_out AS DATE) AS check_out,
       guests_count, total_amount, status,
       CAST(created_at AS TIMESTAMP) AS created_at,
       CAST(updated_at AS TIMESTAMP) AS updated_at,
       'batch'    AS origen
FROM bronze.bronze_bookings
UNION ALL
SELECT booking_id, user_id, property_id,
       CAST(check_in AS DATE)  AS check_in,
       CAST(check_out AS DATE) AS check_out,
       guests_count, total_amount, status,
       CAST(created_at AS TIMESTAMP) AS created_at,
       CAST(updated_at AS TIMESTAMP) AS updated_at,
       'streaming' AS origen
FROM bronze.bronze_bookings_stream;

SELECT origen, COUNT(*) AS registros
FROM bronze.bronze_bookings_all
GROUP BY origen
ORDER BY origen;

## Conclusión

El notebook demuestra el flujo completo de streaming exigido por la guía:

1. **Productor** sintético basado en `rate` source genera eventos de reserva
   coherentes con los rangos de IDs reales en Silver.
2. **Spark Structured Streaming** procesa el stream con `readStream` y aplica
   transformaciones declarativas con `selectExpr`.
3. Los eventos se persisten en una tabla Delta `bronze.bronze_bookings_stream`
   con `checkpointLocation` para garantizar exactly-once.
4. Una vista `bronze_bookings_all` une la carga batch original con los eventos
   de streaming, lo que permite que las capas Silver y Gold consuman la fuente
   combinada sin cambios.

## Decisiones de diseño defendibles ante el docente

- **¿Por qué `rate` source y no Kafka?** Confluent Cloud y plataformas
  equivalentes exigen método de pago. La técnica de Structured Streaming
  (DataFrame API, checkpointing, micro-batching, output modes) es idéntica
  con cualquier fuente. Migrar a Kafka es reemplazar 3 líneas del `readStream`.
- **¿Por qué `checkpointLocation`?** Garantiza exactly-once y permite reanudar
  el stream tras un fallo sin duplicar eventos. Es requisito de Delta como sink.
- **¿Por qué `trigger(processingTime="5 seconds")`?** Micro-batches cortos para
  baja latencia, sin la complejidad operacional del continuous processing.
- **¿Por qué `rand()` con rangos de Silver?** Los eventos sintéticos respetan
  los rangos de claves reales para que los joins con `silver_users` y
  `silver_properties` no produzcan huérfanos.